In [1]:
!python3 -m pip install pyarango
!python3 -m pip install "python-arango>=5.0" 


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [8]:
import json
import requests
import sys
import time                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

from arango.client import ArangoClient

In [15]:

# Initialize the client for ArangoDB.
client = ArangoClient(hosts="http://localhost:8529")


# Connect to "db_test" database as root user.
db = client.db("db_test", username="root", password="test")       

aql = db.aql

<StandardCollection hostname>


## Create a collection

In [19]:
# create a collection test if not exist
if db.has_collection(name="Characters"):
    pass
else:
    db.create_collection(name="Characters")

## Creating and Reading Documents

The syntax for creating a new document is `INSERT document INTO collectionName`. The document is an object like you may know it from JavaScript or JSON, which is comprised of attribute key and value pairs. The quotes around the attribute keys are optional in AQL. Keys are always character sequences (strings), whereas attribute values can have different types:

 - null
 - boolean (true, false)
 - number (integer and floating point)
 - string
 - array
 - object
 
Name and surname of the character document we inserted are both string values. The alive state uses a boolean. Age is a numeric value. The traits are an array of strings. The entire document is an object.

In [20]:
insert_query = """
INSERT {
    "name": "Ned",
    "surname": "Stark",
    "alive": true,
    "age": 41,
    "traits": ["A","H","C","N","P"]
} INTO Characters
"""

# python-arango
aql.execute(insert_query)

<Cursor>

Let us check whether the insert was sucessfull querying the `Characters` collections. The syntax of the loop is `FOR variableName IN collectionName`. For each document in the collection, c is assigned a document, which is then returned as per the loop body.

In [36]:
all_characters = """
FOR c IN Characters
    RETURN c
"""

query_result = aql.execute(all_characters)



if (query_result != None):
    for doc in  query_result:
        print(doc)
        print()

{'_key': '17371', '_id': 'Characters/17371', '_rev': '_ldtSjRW---', 'name': 'Ned', 'surname': 'Stark', 'alive': True, 'age': 41, 'traits': ['A', 'H', 'C', 'N', 'P']}

{'_key': '17921', '_id': 'Characters/17921', '_rev': '_ldtkmay---', 'name': 'Robert', 'surname': 'Baratheon', 'alive': False, 'traits': ['A', 'H', 'C']}

{'_key': '17922', '_id': 'Characters/17922', '_rev': '_ldtkmay--_', 'name': 'Jaime', 'surname': 'Lannister', 'alive': True, 'age': 36, 'traits': ['A', 'F', 'B']}

{'_key': '17923', '_id': 'Characters/17923', '_rev': '_ldtkmay--A', 'name': 'Catelyn', 'surname': 'Stark', 'alive': False, 'age': 40, 'traits': ['D', 'H', 'C']}

{'_key': '17924', '_id': 'Characters/17924', '_rev': '_ldtkmay--B', 'name': 'Cersei', 'surname': 'Lannister', 'alive': True, 'age': 36, 'traits': ['H', 'E', 'F']}

{'_key': '17925', '_id': 'Characters/17925', '_rev': '_ldtkmay--C', 'name': 'Daenerys', 'surname': 'Targaryen', 'alive': True, 'age': 16, 'traits': ['D', 'H', 'C']}

{'_key': '17926', '_id':

In [30]:
insert_query = """
LET data = [
    { "name": "Robert", "surname": "Baratheon", "alive": false, "traits": ["A","H","C"] },
    { "name": "Jaime", "surname": "Lannister", "alive": true, "age": 36, "traits": ["A","F","B"] },
    { "name": "Catelyn", "surname": "Stark", "alive": false, "age": 40, "traits": ["D","H","C"] },
    { "name": "Cersei", "surname": "Lannister", "alive": true, "age": 36, "traits": ["H","E","F"] },
    { "name": "Daenerys", "surname": "Targaryen", "alive": true, "age": 16, "traits": ["D","H","C"] },
    { "name": "Jorah", "surname": "Mormont", "alive": false, "traits": ["A","B","C","F"] },
    { "name": "Petyr", "surname": "Baelish", "alive": false, "traits": ["E","G","F"] },
    { "name": "Viserys", "surname": "Targaryen", "alive": false, "traits": ["O","L","N"] },
    { "name": "Jon", "surname": "Snow", "alive": true, "age": 16, "traits": ["A","B","C","F"] },
    { "name": "Sansa", "surname": "Stark", "alive": true, "age": 13, "traits": ["D","I","J"] },
    { "name": "Arya", "surname": "Stark", "alive": true, "age": 11, "traits": ["C","K","L"] },
    { "name": "Robb", "surname": "Stark", "alive": false, "traits": ["A","B","C","K"] },
    { "name": "Theon", "surname": "Greyjoy", "alive": true, "age": 16, "traits": ["E","R","K"] },
    { "name": "Bran", "surname": "Stark", "alive": true, "age": 10, "traits": ["L","J"] },
    { "name": "Joffrey", "surname": "Baratheon", "alive": false, "age": 19, "traits": ["I","L","O"] },
    { "name": "Sandor", "surname": "Clegane", "alive": true, "traits": ["A","P","K","F"] },
    { "name": "Tyrion", "surname": "Lannister", "alive": true, "age": 32, "traits": ["F","K","M","N"] },
    { "name": "Khal", "surname": "Drogo", "alive": false, "traits": ["A","C","O","P"] },
    { "name": "Tywin", "surname": "Lannister", "alive": false, "traits": ["O","M","H","F"] },
    { "name": "Davos", "surname": "Seaworth", "alive": true, "age": 49, "traits": ["C","K","P","F"] },
    { "name": "Samwell", "surname": "Tarly", "alive": true, "age": 17, "traits": ["C","L","I"] },
    { "name": "Stannis", "surname": "Baratheon", "alive": false, "traits": ["H","O","P","M"] },
    { "name": "Melisandre", "alive": true, "traits": ["G","E","H"] },
    { "name": "Margaery", "surname": "Tyrell", "alive": false, "traits": ["M","D","B"] },
    { "name": "Jeor", "surname": "Mormont", "alive": false, "traits": ["C","H","M","P"] },
    { "name": "Bronn", "alive": true, "traits": ["K","E","C"] },
    { "name": "Varys", "alive": true, "traits": ["M","F","N","E"] },
    { "name": "Shae", "alive": false, "traits": ["M","D","G"] },
    { "name": "Talisa", "surname": "Maegyr", "alive": false, "traits": ["D","C","B"] },
    { "name": "Gendry", "alive": false, "traits": ["K","C","A"] },
    { "name": "Ygritte", "alive": false, "traits": ["A","P","K"] },
    { "name": "Tormund", "surname": "Giantsbane", "alive": true, "traits": ["C","P","A","I"] },
    { "name": "Gilly", "alive": true, "traits": ["L","J"] },
    { "name": "Brienne", "surname": "Tarth", "alive": true, "age": 32, "traits": ["P","C","A","K"] },
    { "name": "Ramsay", "surname": "Bolton", "alive": true, "traits": ["E","O","G","A"] },
    { "name": "Ellaria", "surname": "Sand", "alive": true, "traits": ["P","O","A","E"] },
    { "name": "Daario", "surname": "Naharis", "alive": true, "traits": ["K","P","A"] },
    { "name": "Missandei", "alive": true, "traits": ["D","L","C","M"] },
    { "name": "Tommen", "surname": "Baratheon", "alive": true, "traits": ["I","L","B"] },
    { "name": "Jaqen", "surname": "H'ghar", "alive": true, "traits": ["H","F","K"] },
    { "name": "Roose", "surname": "Bolton", "alive": true, "traits": ["H","E","F","A"] },
    { "name": "The High Sparrow", "alive": true, "traits": ["H","M","F","O"] }
]

FOR d IN data
    INSERT d INTO Characters
"""

aql.execute(insert_query)



<Cursor>

## Updating Documents
### Spoiler Warning!

According to our Ned Stark document, he is alive. When we get to know that he died, we need to change the alive attribute. Let us modify the existing document. For this we first identify the above _key attribute.

In [37]:
find_ned_query = """
FOR c IN Characters
    FILTER c.name == "Ned"
    RETURN c._key
"""

neds_document_key = None

query_result = aql.execute(find_ned_query)

for doc in query_result:
    print("_key: " + str(doc))
    neds_document_key = doc
    print()

_key: 17371



Using `key`we can update an existing document:

`batch_size` et `bind_vars`

 - `batch_size=1`
C'est le nombre de résultats retournés par lot depuis le serveur.

    - ArangoDB ne retourne pas forcément tous les résultats d'un coup — il les pagine en "curseurs"
    - batch_size=1 → le serveur envoie 1 document à la fois
    - Utile pour les grosses requêtes afin d'éviter de surcharger la mémoire
    - Pour une simple mise à jour comme kill_ned, ça n'a pas vraiment d'impact pratique

 - `bind_vars=bindVars`
Ce sont les variables liées à la requête AQL — l'équivalent des paramètres préparés en SQL.

    - Évite les injections AQL
    - Réutilisation de la même requête avec des valeurs différentes
    - Code plus lisible

In [38]:
kill_ned = """
UPDATE @key 
WITH { alive: false} 
IN Characters
"""
bindVars = {'key': neds_document_key}

aql.execute(kill_ned, batch_size=1, bind_vars=bindVars)

find_ned_query = """
FOR c IN Characters
    FILTER c.name == "Ned"
    RETURN c
"""

query_result = aql.execute(find_ned_query)

for doc in  query_result:
    print(doc)
    print()

{'_key': '17371', '_id': 'Characters/17371', '_rev': '_lduFBg----', 'name': 'Ned', 'surname': 'Stark', 'alive': False, 'age': 41, 'traits': ['A', 'H', 'C', 'N', 'P']}



We could have also replaced the entire document content, using `REPLACE` instead of `UPDATE`:


In [39]:
kill_ned = """
REPLACE @key WITH {
    name: "Ned",
    surname: "Stark",
    alive: false,
    age: 41,
    traits: ["A","H","C","N","P"]
} IN Characters
"""
bindVars = {'key': neds_document_key}

aql.execute(kill_ned, bind_vars=bindVars)

find_ned_query = """
FOR c IN Characters
    FILTER c.name == "Ned"
    RETURN c
"""

query_result = aql.execute(find_ned_query)

for doc in  query_result:
    print(doc)
    print()

{'_key': '17371', '_id': 'Characters/17371', '_rev': '_lduGLtS---', 'age': 41, 'alive': False, 'name': 'Ned', 'surname': 'Stark', 'traits': ['A', 'H', 'C', 'N', 'P']}



## Delete Documents

To fully remove documents from a collection, there is the `REMOVE` operation. It works similar to the other modification operations, yet without a WITH clause:

In [40]:
remove_ned = """
REMOVE @key IN Characters
"""
bindVars = {'key': neds_document_key}


try:
    aql.execute(remove_ned, bind_vars=bindVars)
except:
    print("Ned already removed")

As you might have already guessed we can again use a `FOR` loop if we want to perform this operation for the entire collection:

In [42]:
remove_all = """
FOR c IN Characters
    REMOVE c IN Characters
"""

aql.execute(remove_all)


all_characters_names = """
FOR c IN Characters
    RETURN c
"""

query_result = aql.execute(all_characters_names, count=True)

if len(query_result) == 0:
    print("No characted left")

No characted left
